# Minimal Event Model for Floating-Body Wave Interaction

## Problem statement

This notebook treats the repository model as a **research instrument**, not a demonstration. The working question is deliberately narrower than the original motivating intuition:

> Under which assumptions does asymmetric exposure to wave events produce an effective attraction between nearby floating bodies?

The effect is allowed to appear, disappear, weaken, or change sign. Null results are acceptable.

## Falsifiable working hypothesis

When two elongated floating bodies are close together, the **effective impulse received on their inward-facing sides** may become smaller than the impulse received on their outward-facing sides. If that happens, the mean lateral net force `F(d)` may become attractive for sufficiently small gap `d`.

This notebook does **not** assume that a lower inner-side hit count must occur, or that any attraction is universal. It tests whether attraction is present in the reduced model, why it appears when it does, and whether the signal survives elementary controls.


## Modeling level and separation of mechanisms

The notebook stays at the reduced 2D event-based Monte Carlo level. Full fluid simulation is intentionally deferred.

Three layers are kept separate throughout:

1. **Geometric shielding**: whether a source-to-side path is blocked or attenuated by the other body.
2. **Wave propagation and attenuation**: reduced radial decay with distance plus a wavelength envelope.
3. **Mechanical response**: only the normal component contributes to side impulse and the gap-closing force.

That separation matters because an apparent attraction could come from shielding, from externally biased forcing, from the force mapping itself, or from unmodeled physics.

## Observables used below

The notebook tracks at least these quantities as functions of gap and modeling choice:

- inner-side versus outer-side hit counts
- inner-side versus outer-side cumulative impulse
- `Delta I(d) = I_outer(d) - I_inner(d)`
- mean gap-closing force `F(d)`
- repeated-update gap evolution `d(t)`
- incidence-angle distributions
- source-contribution maps in the plane


In [ ]:
from dataclasses import replace
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.append(str(SRC_ROOT))

from float_sim.event_model import (
    ModelParameters,
    ShieldingModel,
    SourceField,
    batch_side_angle_histogram,
    batch_source_contribution_map,
    event_count_from_source_density,
    plot_contribution_map,
    plot_ensemble_metric,
    plot_gap_trajectory_ensemble,
    plot_geometry,
    plot_inner_outer_summary,
    plot_side_metrics,
    run_gap_ensemble_sweep,
    simulate_batch,
    simulate_gap_trajectory_ensemble,
)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
np.set_printoptions(precision=3, suppress=True)


In [ ]:
BASE_PARAMS = ModelParameters(
    body_length=3.0,
    body_width=0.4,
    domain_half_length=6.0,
    domain_half_width=4.0,
    attenuation_length=2.5,
    attenuation_power=0.0,
    characteristic_wavelength=4.0,
    mean_wave_amplitude=1.0,
    side_samples=17,
    mobility=0.0015,
)

GAPS = np.array([0.1, 0.3, 0.6, 1.0, 1.4, 1.8])
BASE_SOURCE_DENSITY = 2.0
ENSEMBLE_REPEATS = 12
TRAJECTORY_REPEATS = 10
BASE_SEED = 20260308
REFERENCE_GAP = 0.6

NO_SHIELDING = ShieldingModel.none()
GRADED_SHIELDING = ShieldingModel.graded(minimum_transmission=0.15, occlusion_decay_length=0.25)

HOMOGENEOUS_FIELD = SourceField(model='uniform')
OUTSIDE_FIELD = SourceField(model='outside_preferred', outside_bias=1.0)

SCENARIOS = {
    'homogeneous / no shielding': (HOMOGENEOUS_FIELD, NO_SHIELDING, '#6c757d'),
    'homogeneous / graded shielding': (HOMOGENEOUS_FIELD, GRADED_SHIELDING, '#1d3557'),
    'outside-preferred / no shielding': (OUTSIDE_FIELD, NO_SHIELDING, '#f4a261'),
    'outside-preferred / graded shielding': (OUTSIDE_FIELD, GRADED_SHIELDING, '#d62828'),
}


def events_for(params: ModelParameters, density: float = BASE_SOURCE_DENSITY) -> int:
    return event_count_from_source_density(density, params)


def run_scenario(
    source_field: SourceField,
    shielding_model: ShieldingModel,
    *,
    params: ModelParameters = BASE_PARAMS,
    gaps: np.ndarray = GAPS,
    density: float = BASE_SOURCE_DENSITY,
    repeats: int = ENSEMBLE_REPEATS,
    seed: int = BASE_SEED,
):
    return run_gap_ensemble_sweep(
        gaps=gaps,
        params=params,
        n_events=events_for(params, density),
        repeats=repeats,
        seed=seed,
        source_field=source_field,
        shielding_model=shielding_model,
    )


def make_summary_rows(label: str, summaries):
    rows = []
    for summary in summaries:
        rows.append({
            'scenario': label,
            'gap': round(summary.gap, 2),
            'F(d)': round(summary.mean_gap_closing_force, 3),
            'Delta I(d)': round(summary.impulse_imbalance, 3),
            'outer-inner hits': round(summary.hit_imbalance, 3),
            'mean inner hits': round(summary.mean_inner_hits, 2),
            'mean outer hits': round(summary.mean_outer_hits, 2),
        })
    return rows


def estimate_sign_change_gap(summaries):
    forces = np.array([summary.mean_gap_closing_force for summary in summaries], dtype=float)
    gaps = np.array([summary.gap for summary in summaries], dtype=float)
    signs = np.sign(forces)
    for left in range(len(gaps) - 1):
        if signs[left] == 0:
            return float(gaps[left])
        if signs[left] != signs[left + 1]:
            return float(0.5 * (gaps[left] + gaps[left + 1]))
    return None


def plot_angle_histogram(ax, histogram, *, label: str, color: str):
    centers = 0.5 * (histogram.bin_edges[:-1] + histogram.bin_edges[1:])
    ax.plot(centers, histogram.weighted_impulse, marker='o', color=color, label=label)
    ax.set_xlabel('Incidence angle (deg)')
    ax.set_ylabel('Impulse-weighted counts')
    ax.set_title('Incidence-angle diagnostic')
    ax.legend(loc='best')


def scenario_force_at_gap(
    *,
    gap: float,
    params: ModelParameters,
    density: float,
    source_field: SourceField,
    shielding_model: ShieldingModel,
    repeats: int = 8,
    seed: int = BASE_SEED,
) -> float:
    summary = run_gap_ensemble_sweep(
        gaps=np.array([gap]),
        params=params,
        n_events=events_for(params, density),
        repeats=repeats,
        seed=seed,
        source_field=source_field,
        shielding_model=shielding_model,
    )[0]
    return float(summary.mean_gap_closing_force)


## Representative batches: what the diagnostics mean

The first figure shows two representative batches at the same gap with graded shielding:

- a **homogeneous** source field
- an **outside-preferred** source field, meant as a reduced external-forcing analogue

For each batch the notebook shows geometry, side hit counts, side cumulative impulse, and impulse-weighted incidence angles. The contribution maps then show where positive `Delta I` contributions come from in the source plane.


In [ ]:
homogeneous_batch = simulate_batch(
    gap=REFERENCE_GAP,
    params=BASE_PARAMS,
    rng=np.random.default_rng(BASE_SEED),
    n_events=events_for(BASE_PARAMS),
    source_field=HOMOGENEOUS_FIELD,
    shielding_model=GRADED_SHIELDING,
)
outside_batch = simulate_batch(
    gap=REFERENCE_GAP,
    params=BASE_PARAMS,
    rng=np.random.default_rng(BASE_SEED),
    n_events=events_for(BASE_PARAMS),
    source_field=OUTSIDE_FIELD,
    shielding_model=GRADED_SHIELDING,
)

homogeneous_inner_angles = batch_side_angle_histogram(homogeneous_batch, BASE_PARAMS, 'inner')
homogeneous_outer_angles = batch_side_angle_histogram(homogeneous_batch, BASE_PARAMS, 'outer')
outside_inner_angles = batch_side_angle_histogram(outside_batch, BASE_PARAMS, 'inner')
outside_outer_angles = batch_side_angle_histogram(outside_batch, BASE_PARAMS, 'outer')

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
plot_geometry(axes[0, 0], homogeneous_batch, max_events=250)
plot_side_metrics(axes[0, 1], homogeneous_batch, metric='hits')
plot_side_metrics(axes[0, 2], homogeneous_batch, metric='impulse')
plot_angle_histogram(axes[0, 3], homogeneous_inner_angles, label='inner', color='#e76f51')
plot_angle_histogram(axes[0, 3], homogeneous_outer_angles, label='outer', color='#1d3557')
axes[0, 0].set_title('Homogeneous / graded shielding')

plot_geometry(axes[1, 0], outside_batch, max_events=250)
plot_side_metrics(axes[1, 1], outside_batch, metric='hits')
plot_side_metrics(axes[1, 2], outside_batch, metric='impulse')
plot_angle_histogram(axes[1, 3], outside_inner_angles, label='inner', color='#e76f51')
plot_angle_histogram(axes[1, 3], outside_outer_angles, label='outer', color='#1d3557')
axes[1, 0].set_title('Outside-preferred / graded shielding')

fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
homogeneous_map = batch_source_contribution_map(homogeneous_batch, BASE_PARAMS, quantity='delta_impulse')
outside_map = batch_source_contribution_map(outside_batch, BASE_PARAMS, quantity='delta_impulse')
image_left = plot_contribution_map(axes[0], homogeneous_map)
image_right = plot_contribution_map(axes[1], outside_map)
axes[0].set_title('Homogeneous: source contributions to Delta I')
axes[1].set_title('Outside-preferred: source contributions to Delta I')
fig.colorbar(image_left, ax=axes[0], shrink=0.8)
fig.colorbar(image_right, ax=axes[1], shrink=0.8)
fig.tight_layout()
plt.show()


## Gap sweeps: does `F(d)` become attractive, and why?

The next sweeps compare two source scenarios and two shielding controls:

- homogeneous forcing with and without shielding
- outside-preferred forcing with and without shielding

The central question is whether attraction is associated with fewer inner hits, lower inner cumulative impulse, or both.


In [ ]:
scenario_summaries = {
    label: run_scenario(source_field, shielding_model)
    for label, (source_field, shielding_model, _) in SCENARIOS.items()
}

fig, axes = plt.subplots(2, 3, figsize=(17, 9), sharex=True)
for label, (_, _, color) in SCENARIOS.items():
    summaries = scenario_summaries[label]
    plot_ensemble_metric(axes[0, 0], summaries, metric='gap_closing_force', label=label, color=color)
    plot_ensemble_metric(axes[0, 1], summaries, metric='impulse_imbalance', label=label, color=color)
    plot_ensemble_metric(axes[0, 2], summaries, metric='hit_imbalance', label=label, color=color)

plot_inner_outer_summary(axes[1, 0], scenario_summaries['homogeneous / graded shielding'], metric='hits')
axes[1, 0].set_title('Homogeneous / graded: inner vs outer hits')
plot_inner_outer_summary(axes[1, 1], scenario_summaries['homogeneous / graded shielding'], metric='impulse')
axes[1, 1].set_title('Homogeneous / graded: inner vs outer impulse')
plot_inner_outer_summary(axes[1, 2], scenario_summaries['outside-preferred / graded shielding'], metric='impulse')
axes[1, 2].set_title('Outside-preferred / graded: inner vs outer impulse')

fig.tight_layout()
plt.show()

for label, summaries in scenario_summaries.items():
    zero_crossing = estimate_sign_change_gap(summaries)
    print(label)
    print('approximate sign-change gap:', None if zero_crossing is None else round(zero_crossing, 2))
    for row in make_summary_rows(label, summaries):
        print(row)
    print()


## Interim interpretation

In this reduced model the baseline story is already informative:

- attraction, when it appears, is better tracked by **cumulative impulse imbalance** than by raw hit counts
- the hit-count diagnostic can stay flat or even favor the inner side while `Delta I(d)` remains positive
- outside-preferred forcing strengthens attraction and tends to push the sign change to larger gaps
- the no-shielding controls are essential because they show that directed forcing alone and shielding alone need to be distinguished, not conflated

This means the original phrase “fewer inner-side wave hits” is too narrow for the present model. The stronger reduced statement is about **effective inner-side impulse**, not just event count.


## Gap evolution under repeated updates

A positive mean force at fixed gap is still weaker evidence than a consistent trajectory tendency. The next figure therefore evolves the gap under repeated reduced-force updates and reports ensemble bands over seeds.


In [ ]:
trajectory_cases = {
    'homogeneous / no shielding': (HOMOGENEOUS_FIELD, NO_SHIELDING, '#6c757d'),
    'homogeneous / graded shielding': (HOMOGENEOUS_FIELD, GRADED_SHIELDING, '#1d3557'),
    'outside-preferred / graded shielding': (OUTSIDE_FIELD, GRADED_SHIELDING, '#d62828'),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
for label, (field, shielding, color) in trajectory_cases.items():
    trajectory = simulate_gap_trajectory_ensemble(
        initial_gap=1.0,
        steps=12,
        n_events_per_step=events_for(BASE_PARAMS, 1.5),
        params=BASE_PARAMS,
        repeats=TRAJECTORY_REPEATS,
        seed=BASE_SEED,
        source_field=field,
        shielding_model=shielding,
    )
    plot_gap_trajectory_ensemble(axes[0], trajectory, label=label, color=color)
    axes[1].plot(
        [point.step for point in trajectory],
        [point.mean_gap_closing_force for point in trajectory],
        marker='o',
        color=color,
        label=label,
    )

axes[1].axhline(0.0, color='0.4', linestyle='--', linewidth=1.0)
axes[1].set_xlabel('Update step')
axes[1].set_ylabel('Mean gap-closing force')
axes[1].set_title('Force during repeated updates')
axes[1].legend(loc='best')
fig.tight_layout()
plt.show()


## Parameter studies and robustness

The reduced model should not be judged on a single hand-picked setting. The next pass varies:

- body length
- body width
- source density
- attenuation length
- wavelength
- shielding strength

Gap dependence is already shown directly in `F(d)`, so the plots below use two reference gaps:

- `d = 0.3` for a short-gap regime
- `d = 1.4` for a wider-gap regime

The goal is not exhaustive coverage. It is to see whether attraction is broad, conditional, or fragile.


In [ ]:
study_gaps = [0.3, 1.4]
parameter_studies = {
    'body_length': [2.0, 3.0, 4.5],
    'body_width': [0.25, 0.4, 0.6],
    'source_density': [1.0, 2.0, 3.0],
    'attenuation_length': [1.5, 2.5, 4.0],
    'characteristic_wavelength': [1.0, 4.0, 8.0],
    'minimum_transmission': [0.0, 0.15, 0.35],
}

parameter_results = {}
for parameter_name, values in parameter_studies.items():
    rows = []
    for value in values:
        if parameter_name == 'source_density':
            params = BASE_PARAMS
            density = value
            shielding = GRADED_SHIELDING
        elif parameter_name == 'minimum_transmission':
            params = BASE_PARAMS
            density = BASE_SOURCE_DENSITY
            shielding = ShieldingModel.graded(minimum_transmission=value, occlusion_decay_length=0.25)
        else:
            params = replace(BASE_PARAMS, **{parameter_name: value})
            density = BASE_SOURCE_DENSITY
            shielding = GRADED_SHIELDING

        for scenario_label, field in [
            ('homogeneous', HOMOGENEOUS_FIELD),
            ('outside-preferred', OUTSIDE_FIELD),
        ]:
            short_force = scenario_force_at_gap(
                gap=study_gaps[0],
                params=params,
                density=density,
                source_field=field,
                shielding_model=shielding,
                repeats=8,
                seed=BASE_SEED,
            )
            wide_force = scenario_force_at_gap(
                gap=study_gaps[1],
                params=params,
                density=density,
                source_field=field,
                shielding_model=shielding,
                repeats=8,
                seed=BASE_SEED,
            )
            rows.append((scenario_label, value, short_force, wide_force))
    parameter_results[parameter_name] = rows

fig, axes = plt.subplots(3, 2, figsize=(15, 12))
axes = axes.ravel()
palette = {'homogeneous': '#1d3557', 'outside-preferred': '#d62828'}
for ax, (parameter_name, rows) in zip(axes, parameter_results.items(), strict=True):
    values = sorted({row[1] for row in rows})
    for scenario_label in ['homogeneous', 'outside-preferred']:
        short_forces = [row[2] for row in rows if row[0] == scenario_label]
        wide_forces = [row[3] for row in rows if row[0] == scenario_label]
        ax.plot(values, short_forces, marker='o', color=palette[scenario_label], label=f'{scenario_label}, d=0.3')
        ax.plot(values, wide_forces, marker='s', linestyle='--', color=palette[scenario_label], label=f'{scenario_label}, d=1.4')
    ax.axhline(0.0, color='0.4', linestyle='--', linewidth=1.0)
    ax.set_title(parameter_name.replace('_', ' '))
    ax.set_xlabel(parameter_name.replace('_', ' '))
    ax.set_ylabel('Mean gap-closing force')
    ax.legend(loc='best', fontsize=8)

fig.tight_layout()
plt.show()

for parameter_name, rows in parameter_results.items():
    print(parameter_name)
    for scenario_label, value, short_force, wide_force in rows:
        print({
            'scenario': scenario_label,
            'value': value,
            'F(0.3)': round(short_force, 3),
            'F(1.4)': round(wide_force, 3),
        })
    print()


## Directed forcing versus shielding: explicit control matrix

The next table is meant to address a common interpretive failure directly. Attraction in the reduced model can be helped by two conceptually different ingredients:

- **externally biased forcing**
- **geometric shielding**

A signal that only appears when both are present should be described as conditional, not general.


In [ ]:
control_gap = 0.6
control_rows = []
for label, (field, shielding, _) in SCENARIOS.items():
    summary = run_gap_ensemble_sweep(
        gaps=np.array([control_gap]),
        params=BASE_PARAMS,
        n_events=events_for(BASE_PARAMS),
        repeats=ENSEMBLE_REPEATS,
        seed=BASE_SEED,
        source_field=field,
        shielding_model=shielding,
    )[0]
    control_rows.append({
        'scenario': label,
        'F(0.6)': round(summary.mean_gap_closing_force, 3),
        'Delta I(0.6)': round(summary.impulse_imbalance, 3),
        'outer-inner hits': round(summary.hit_imbalance, 3),
    })

for row in control_rows:
    print(row)


## Alternative explanations and current limits

This notebook can separate some explanations and only discuss others qualitatively.

### What the reduced model can test directly

- whether shielding changes **effective side exposure**
- whether `Delta I(d)` tracks `F(d)` better than raw hit counts
- whether homogeneous and outside-preferred forcing differ materially
- whether the effect is sensitive to wavelength, attenuation, geometry, or shielding strength

### What the reduced model cannot settle cleanly

- **interference**: the model uses independent events, not a phase-resolved field
- **hydrodynamic coupling and displacement flow**: body-induced fluid motion is omitted
- **capillary or meniscus effects**: not represented at all
- **reflection**: only reduced transmission is modeled; explicit reflection has not been added yet

These are real limitations, not bookkeeping details. If the event-based effect is fragile, the next step is not to hide that fragility with complexity.


## Preparing for second-stage validation

The helper code is now organized so that a later continuous-field validation layer can be added without replacing the event model.

A plausible future extension is a damped height field `h(x, y, t)` on a grid, schematically

`d²h/dt² = c² ∇²h - beta * dh/dt + S(x, y, t)`

but that remains a **second-stage check**, not the primary instrument. The notebook stays focused on the interpretable event model until its behavior is mapped clearly enough to justify that extra layer.


## Conclusions from this iteration

The reduced model does **not** support a blanket statement that nearby bodies simply receive fewer inner-side hits and therefore always attract.

What the present notebook is designed to check more carefully is this:

- whether attraction appears in `F(d)` at small gap
- whether that attraction weakens or changes sign as gap grows
- whether the signal is better explained by `Delta I(d)` than by hit-count asymmetry
- whether the signal survives under homogeneous forcing, or depends mainly on outside-preferred forcing and shielding

In the baseline runs used here, the most important working interpretation is expected to be:

- inner-side **impulse** reduction is more informative than inner-side **hit-count** reduction
- attraction is conditional rather than universal
- outside-preferred forcing generally strengthens the effect and extends it to larger gaps
- homogeneous forcing can still show attraction with shielding at small gaps, but the effect weakens and can change sign as the gap grows

That is a conditional reduced-model result, not a proof about real floating bodies.
